In [11]:
from bm25 import BM25Retriever
from vdb import FaissRetriever

In [12]:
NUM_CHAPTERS = 10
faiss_retriever = FaissRetriever(chunks_file=f'first{NUM_CHAPTERS}_chunks.json')
bm25_retriever = BM25Retriever(chunks_file=f'first{NUM_CHAPTERS}_chunks.json')

query = "杨奇从城主府偷走了什么？"
faiss_results = faiss_retriever.retrieve(query, top_k=5)
bm25_results = bm25_retriever.retrieve(query, top_k=5)
print("FAISS 检索结果:")
for res in faiss_results:
    print(f"Chapter ID: {res['chpt_id']}, Chunk ID: {res['chunk_id']}, Score: {res['score']}, Content: {res['text_snippet']}\n")
print("BM25 检索结果:")
for res in bm25_results:
    print(f"Chapter ID: {res['chpt_id']}, Chunk ID: {res['chunk_id']}, Score: {res['score']}, Content: {res['text_snippet']}\n")

已加载 FAISS 索引，包含 62 个向量。
FAISS 检索结果:
Chapter ID: 2, Chunk ID: 2, Score: 0.6477575898170471, Content: “是的，而且城主府邸明确的下达了最后通牒，三天之内，要杨家拿出赔偿被盗窃掉的宝贝损失方案，一个月内必须全部付清。这样一来，杨家最少要损失一大半的产业！”一个稳重的仆人忧心忡忡的道。
顿时，许多的仆人丫鬟都发出了一声轰鸣，蜩螗羹沸似的爆发了：“损失一大半的产业？那杨家不是衰落下去了么？”
“尤其是在这燕都城中，还有陈，王，李，洪……这些豪门，早就对我们杨家不顺眼，肯定会落井下石吧。”
“看看吧，这次杨家麻烦大了，杨奇那个纨绔真是不让人省心，闯出来了这样的弥天大祸，嘿嘿，我看他父亲杨战连家主的位置都坐不稳当了。”
“我今天听城主府的熟人说，那杨奇是为了一个女人，才偷窃的宝贝。”
“呸，纨绔，祸根，害人精。”一个仆人暗中骂道：“被废了气功，从此之后没有武功，就算好了也是一个残废。”
这些仆人，有的幸灾乐祸，有的担心杨家的未来，有的尖刻诅咒杨奇。
此时，在整个庭院中心，经过九曲回廊迷宫一般的长廊屋檐，宽阔的杨家议事厅中，一个高大，魁梧，身穿锦衣的男子站立在中央，听着汇报。
“老爷，杨奇少爷的性命已经保住了，不过全身被雷电大面积烧伤，体内经脉破损，气海被破，武功全废，从此之后只能够变成残废了。”
一个老管家带着医师在缓缓的汇报着。
“武功全废，气海被破！”杨战的嘴里重复着这两句话。
砰！
巨掌拍下。
一张铁木大桌粉碎。
那碎木屑都疯狂旋转，呜呜呜鬼哭神嚎一般飞了出去，镶落在地面上。
“丢了伏龙丹！我们杨家可以赔偿，为什么要废掉我儿子的气功？哪怕是断他的手脚都可以。废了武功，一辈子都不能够凝聚气功，断掉手脚，气功还在，一样能够成为高手。”杨战怒不可遏。

Chapter ID: 5, Chunk ID: 2, Score: 0.7069646120071411, Content: 端坐在自己房屋中，他再次进入了深沉的冥想，揣摩神象镇狱劲的奥秘……
第三天一大早，天刚蒙蒙亮。
整个燕都城的八个城门口，就出现了许多马车，一支支的商队，都赶向了燕都城中的杨家府邸，这让许多人都为之侧目。
“出了什么事？怎么这么多的商队都到杨家去了？这些人是什么人？”
“大约你还不知道吧，这些都是杨家在另外

In [13]:
def rrf_fusion(faiss_results, bm25_results, k=60, top_k=5):
    scores = {}
    visit = set()
    fusion_results = []
    def update_scores(results):
        # rank从0开始，会不会有问题？
        for rank, res in enumerate(results):
            key = (res['chpt_id'], res['chunk_id'])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            if key not in visit:
                visit.add(key)
                fusion_results.append({
                    "chpt_id": res['chpt_id'],
                    "chunk_id": res['chunk_id'],
                    "score": -1,  # 占位，稍后更新
                    "text_snippet": res['text_snippet']
                })
    update_scores(faiss_results)
    update_scores(bm25_results)
    for res in fusion_results:
        key = (res['chpt_id'], res['chunk_id'])
        res['score'] = scores[key]
    return sorted(fusion_results, key=lambda x: x['score'], reverse=True)[:top_k]

In [14]:
print("RRF 融合结果:")
fused_results = rrf_fusion(faiss_results, bm25_results, k=60, top_k=5)
for res in fused_results:
    print(f"Chapter ID: {res['chpt_id']}, Chunk ID: {res['chunk_id']}, Score: {res['score']}, Content: {res['text_snippet']}\n")

RRF 融合结果:
Chapter ID: 2, Chunk ID: 2, Score: 0.03306010928961749, Content: “是的，而且城主府邸明确的下达了最后通牒，三天之内，要杨家拿出赔偿被盗窃掉的宝贝损失方案，一个月内必须全部付清。这样一来，杨家最少要损失一大半的产业！”一个稳重的仆人忧心忡忡的道。
顿时，许多的仆人丫鬟都发出了一声轰鸣，蜩螗羹沸似的爆发了：“损失一大半的产业？那杨家不是衰落下去了么？”
“尤其是在这燕都城中，还有陈，王，李，洪……这些豪门，早就对我们杨家不顺眼，肯定会落井下石吧。”
“看看吧，这次杨家麻烦大了，杨奇那个纨绔真是不让人省心，闯出来了这样的弥天大祸，嘿嘿，我看他父亲杨战连家主的位置都坐不稳当了。”
“我今天听城主府的熟人说，那杨奇是为了一个女人，才偷窃的宝贝。”
“呸，纨绔，祸根，害人精。”一个仆人暗中骂道：“被废了气功，从此之后没有武功，就算好了也是一个残废。”
这些仆人，有的幸灾乐祸，有的担心杨家的未来，有的尖刻诅咒杨奇。
此时，在整个庭院中心，经过九曲回廊迷宫一般的长廊屋檐，宽阔的杨家议事厅中，一个高大，魁梧，身穿锦衣的男子站立在中央，听着汇报。
“老爷，杨奇少爷的性命已经保住了，不过全身被雷电大面积烧伤，体内经脉破损，气海被破，武功全废，从此之后只能够变成残废了。”
一个老管家带着医师在缓缓的汇报着。
“武功全废，气海被破！”杨战的嘴里重复着这两句话。
砰！
巨掌拍下。
一张铁木大桌粉碎。
那碎木屑都疯狂旋转，呜呜呜鬼哭神嚎一般飞了出去，镶落在地面上。
“丢了伏龙丹！我们杨家可以赔偿，为什么要废掉我儿子的气功？哪怕是断他的手脚都可以。废了武功，一辈子都不能够凝聚气功，断掉手脚，气功还在，一样能够成为高手。”杨战怒不可遏。

Chapter ID: 2, Chunk ID: 1, Score: 0.03279569892473118, Content: 另一个侍卫立刻拔腿狂奔，如烈马疾驰。
但是，他才冲出了一里路程，就被一尊钢铁巨人提了回来。
“罗魂大人。”这个侍卫吓出来了一身冷汗，看清楚了来人，立刻噗通跪在泥水之中。
“居然被雷劈中了……”钢铁巨兽似的罗魂大步走到了杨奇面前，伸手一抓，一股气流渗透进入了他的身躯，却什么都没有发现，只看见了残破的经脉，气海，还有焦糊的身躯，“他的身躯，遭遇到雷劈，创伤